# Set up environment and load data

In [2]:
import pandas as pd
import numpy as np

files = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv"
]

df_list = [pd.read_csv(f) for f in files]

df = pd.concat(df_list, ignore_index=True)

print(df.shape)
print(df.columns.tolist())

(991847, 79)
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count'

# Clean column names

In [3]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

# Check label distribution

In [4]:
print(df['Label'].value_counts())

Label
BENIGN          702888
PortScan        158930
DDoS            128027
Bot               1966
Infiltration        36
Name: count, dtype: int64


# Handle missing and infinite values

In [5]:
# Replace infinities with NaN, then check how many
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.isna().sum().sum(), "missing values found")

# Drop rows with missing values (safe since it's usually a small %)
df.dropna(inplace=True)
print(df.shape)

1468 missing values found
(991113, 79)


# Remove duplicate columns

In [6]:
# Check for duplicate columns
print(df.columns[df.columns.duplicated()])

# Check for constant columns (only 1 unique value = useless)
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns:", constant_cols)

df.drop(columns=constant_cols, inplace=True)

Index([], dtype='str')
Constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


# Encode the target label

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Save this mapping — you'll need it later to interpret predictions
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

{'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'Infiltration': np.int64(3), 'PortScan': np.int64(4)}


# Separate features (X) and target (y)

In [8]:
X = df.drop(columns=['Label', 'Label_encoded'])
y = df['Label_encoded']

print(X.shape, y.shape)

(991113, 70) (991113,)


# Feature selection

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Use a small sample first (this dataset is huge — don't run full data yet)
sample_X = X.sample(50000, random_state=42)
sample_y = y.loc[sample_X.index]

rf_temp = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_temp.fit(sample_X, sample_y)

importances = pd.Series(rf_temp.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(20)
print(top_features)

Packet Length Variance         0.078105
Total Length of Fwd Packets    0.071314
Subflow Fwd Bytes              0.060675
Packet Length Mean             0.059560
Fwd Packet Length Mean         0.054078
Bwd Packet Length Std          0.052681
Fwd Packet Length Max          0.043822
Packet Length Std              0.039852
Bwd Packet Length Mean         0.032707
Bwd Packets/s                  0.031384
Total Length of Bwd Packets    0.031261
Avg Fwd Segment Size           0.030055
Avg Bwd Segment Size           0.029323
Average Packet Size            0.026626
act_data_pkt_fwd               0.025801
Total Fwd Packets              0.024302
Subflow Fwd Packets            0.023027
Fwd Header Length              0.020993
Init_Win_bytes_forward         0.019212
Destination Port               0.018388
dtype: float64


In [10]:
selected_features = top_features.index.tolist()
X_selected = X[selected_features]

# Feature scaling

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

In [13]:
%pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
Using cached imbalanced_learn-0.14.2-py3-none-any.whl (236 kB)
Note: you may need to restart the kernel to use updated packages.


# Handle class imbalance

In [14]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=3)  # small k because Infiltration has very few samples
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print(pd.Series(y_resampled).value_counts())

Label_encoded
0    702292
2    702292
4    702292
1    702292
3    702292
Name: count, dtype: int64


# Train-test split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print(X_train.shape, X_test.shape)

(2809168, 20) (702292, 20)


In [16]:
# Combine cleaned features + original label back together for a readable saved file
df_cleaned = X.copy()
df_cleaned['Label'] = df['Label']   # keep original text label, not encoded

df_cleaned.to_csv("cleaned_dataset.csv", index=False)
print("Saved cleaned_dataset.csv with shape:", df_cleaned.shape)

Saved cleaned_dataset.csv with shape: (991113, 71)


In [17]:
df_selected = X_selected.copy()
df_selected['Label'] = df['Label'].values

df_selected.to_csv("feature_selected_dataset.csv", index=False)
print("Saved feature_selected_dataset.csv with shape:", df_selected.shape)

Saved feature_selected_dataset.csv with shape: (991113, 21)


In [18]:
# Convert back to DataFrame since SMOTE/scaler output numpy arrays
final_df = pd.DataFrame(X_resampled, columns=selected_features)
final_df['Label_encoded'] = y_resampled

final_df.to_csv("final_training_ready_dataset.csv", index=False)
print("Saved final_training_ready_dataset.csv with shape:", final_df.shape)

Saved final_training_ready_dataset.csv with shape: (3511460, 21)


In [20]:
import json

# Convert NumPy values to normal Python types
label_mapping_clean = {
    str(k): int(v) if hasattr(v, "item") else v
    for k, v in label_mapping.items()
}

with open("label_mapping.json", "w") as f:
    json.dump(label_mapping_clean, f, indent=4)

print("Saved label_mapping.json:")
print(label_mapping_clean)

Saved label_mapping.json:
{'BENIGN': 0, 'Bot': 1, 'DDoS': 2, 'Infiltration': 3, 'PortScan': 4}


In [21]:
import joblib

joblib.dump(scaler, "scaler.pkl")
joblib.dump(selected_features, "selected_features.pkl")

print("Saved scaler.pkl and selected_features.pkl")

Saved scaler.pkl and selected_features.pkl
